# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadhany222/flyrank-ml-assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a **ranking / scoring** task. My Lane 2 question is "which pages should a reviewer
look at first" — that's not "yes/no, is this page bad" (classification) and it's not "what
groups exist" (clustering). It's about ORDER: given limited reviewer time, which pages come
first. The output I need is a score per page that produces a ranked list, not a single
category label. This matches the framing-ml-problems mapping directly: "which ones first?"
→ ranking/scoring, with precision@K as the natural metric.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


For now I'm using the starter's existing proxy label: `trend_direction == "down"`. This
comes from a **defined rule inside the current window** — it's calculated from where a
page's trend sits right now, not from what actually happens to it next. That makes it a
proxy, not a true observed outcome.

The stronger version — which I'd move to for the capstone — is a real future-window label:
features from the prior 90 days predicting an actual decline over the next 30 days. I'm not
building that yet since it needs the warehouse's daily fact table (`fact_content_daily_performance`)
to construct real time windows, which I don't have loaded yet. For this assignment I'm being
explicit that today's target is a proxy, exactly so I don't accidentally treat it as more
than it is.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50.** A reviewer can realistically get through about 50 pages in a review cycle,
so what matters is: of the top 50 pages my ranking puts first, how many are actually
positive (declining) by the label? That's a number I can directly defend against real
review capacity — unlike overall accuracy, which would reward correctly ignoring the
thousands of pages nobody was going to look at anyway.

I already have a number to compare against: the starter's hand-rule baseline gets
Precision@50 = 0.240, and a learned model in the reference pipeline gets up to 0.740 on
this same slice. "Good" for my lane means beating 0.240 by a real, honestly-validated margin.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page, filtered down to my lane's slice: pages with real visibility
(`impressions_90d >= 100`), same definition I used in w01. That's the population a reviewer
would actually be choosing among — pages nobody sees aren't candidates for review at all.

In [5]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/muhammadhany222/flyrank-ml-assignment1"  # your repo
REPO_DIR = "flyrank-ml-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

In [6]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Apply the lane's slice: pages with real visibility (per w01 framing)
df = df[df["impressions_90d"] >= 100].copy()
print(f"Lane slice: {df.shape[0]} rows (visible pages only)")

# One row = one content page (already deduplicated by content_id in the starter prep)
print(f"One row = one page. Unique content_ids: {df['content_id'].nunique()}")

df[["content_id", "client_id", "impressions_90d", "days_since_last_update",
    "avg_position", "ctr", "trend_direction"]].head(10)

Lane slice: 22006 rows (visible pages only)
One row = one page. Unique content_ids: 22006


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,down
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,down
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,down
5,content_d4084a4bc775,client_f369cb89fc,3970,20,8.5,0.03,down
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,0.06,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,20,46.0,0.09,down
9,content_c27558df2b0c,client_19581e27de,1240,104,4.9,0.16,down
10,content_d8ee6cc6d642,client_19581e27de,20919,104,2.2,1.55,stable


In [7]:
# Sketch of what the target column looks like
target = (df["trend_direction"] == "down").astype(int)
print("Target value counts (1 = declining, 0 = not):")
print(target.value_counts())
print(f"\nBase rate of positive class: {target.mean():.1%}")

Target value counts (1 = declining, 0 = not):
trend_direction
1    13152
0     8854
Name: count, dtype: int64

Base rate of positive class: 59.8%


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


I already tested this directly in notebook 02. My hand rule ("stale AND visible") only
checks one interaction between two columns, and when I applied it to this same data it
matched only 35 pages out of 22,006 visible ones (0.2%) — far too strict to be a usable
review queue. A depth-3 decision tree, trained on the same signals, found Precision@50 =
0.72 versus the hand rule's 0.24 in the reference pipeline — nearly 3x better — by
combining multiple signals (impressions, position, freshness, content age, CTR) in ways
that shift depending on their combination, not a single fixed threshold.

The pattern is too messy for an if-statement because "declining" isn't caused by any one
signal alone — it's a mix of how visible a page is, how stale it's gotten, where it ranks,
and how well it converts impressions into clicks, and those interact differently across
different content types and position tiers. A hand rule freezes one guess about how they
interact; a model can find the actual shape of that interaction from the data itself.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.